# German Replication of Greenwood, Shleifer & You (2019)

This notebook is a reporting and reproducibility layer for the German replication/adaptation of "Bubbles for Fama." The raw-data pipeline was executed in Stata; the objective here is to read the executed project files and present the results in a professional seminar-paper format.

The notebook is designed to be portable: it locates the project directory from the current working directory and then reads the executed Excel files and images in that folder.


## 1. Project Setup

The notebook begins from a clean project directory and reads the executed outputs rather than recreating the raw-data calculations. The analysis is restricted to Germany and follows the replication logic of Greenwood, Shleifer, and You (2019) as closely as the available WRDS extract permits.


In [ ]:
from pathlib import Path
import json
import re
from typing import Optional

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

EXPECTED_FILES = [
    'germany_crash_results.xlsx',
    'germany_crash_test.xlsx',
    'germany_footnote15_regressions.xlsx',
    'germany_event_time_paths.xlsx',
    'germany_table1_paper_style.xlsx',
    'germany_table3_main.xlsx',
    'germany_replication_checks.xlsx',
    'germany_event_time_paths.png',
]

def find_project_dir() -> Path:
    cwd = Path.cwd()
    candidates = []

    # Current directory and parent directories
    for base in [cwd, *list(cwd.parents)]:
        candidates.append(base)
        candidates.extend(base.iterdir())

    # Common child directories
    for base in [cwd, *list(cwd.parents)]:
        for child in ['data', 'project', 'results', 'files', 'input', 'inputs']:
            candidate = base / child
            if candidate.exists():
                candidates.append(candidate)

    scored = []
    for candidate in dict.fromkeys(candidates):
    if not candidate.exists() or not candidate.is_dir():
        continue
    present = sum(1 for name in EXPECTED_FILES if (candidate / name).exists())
    score = present * 100
    if (candidate / 'germany_crash_results.xlsx').exists():
        score += 10
    if (candidate / 'germany_event_time_paths.png').exists():
        score += 10
    scored.append((score, candidate, present))

    if not scored:
        raise FileNotFoundError(
            'Could not identify the project folder. Expected executed files were not found in the current working directory or nearby parent folders.'
        )

    scored.sort(key=lambda x: (-x[0], -x[2], str(x[1])))
    best_score, best_dir, _ = scored[0]
    if best_score <= 0:
        raise FileNotFoundError(
            'No project directory matched the expected Germany replication result files. Place the executed files in the project folder or run the notebook from that directory.'
        )
    return best_dir

PROJECT_DIR = find_project_dir()
PROJECT_DIR


## 2. Portable Project-File Discovery

The project folder is discovered portably using `pathlib` and the presence of the executed German replication files. The notebook does not rely on a machine-specific absolute path and can be copied to another PC as long as the project files remain in the same project folder.


In [ ]:
display(Markdown(f'**Project directory:** `{PROJECT_DIR}`'))
for name in EXPECTED_FILES:
    p = PROJECT_DIR / name
    print(f'{name}: {"FOUND" if p.exists() else "MISSING"}')


## 3. Research Design and Data Constraint

The empirical design follows the logic of the original Greenwood, Shleifer, and You (2019) study: identify large industry run-ups, study subsequent crash risk and drawdown behavior, evaluate raw and net-of-market returns, and use event-time analysis around episodes. The German replication is deliberately narrower than the original paper because the project is restricted to one WRDS source.

The notebook therefore reads the executed Stata outputs rather than recomputing the full raw-data pipeline in Python. The raw WRDS file (`comp_global_daily.dta`) remains the authoritative data source for the underlying calculations, while this notebook provides the presentation and reproducibility layer.


## 4. Source Data and Sample Construction

The Germany replication uses only the supplied analytical file and applies the country filter `fic == "DEU"`. The sample is industry-level and uses the primary GICS sector classification (`gsector`), while retaining `ggroup`, `gind`, and `gsubind` for additional structure. Security-level identifiers are `gvkey + iid`, and common equity is selected with `tpci == "0"`.

The project was constructed from a Germany-only extract in the supplied WRDS data, with the daily security sample then reduced by eligibility screens and daily return cleaning. The final executed files reflect the cleaned and aggregated analytical datasets rather than a fresh raw-data reprocessing in Python.


## 5. Daily Security Cleaning and Currency Normalization

The security-level total-return price is constructed as `(prccd * fx_eur / ajexdi) * trfd`, where the reporting currency is EUR. Both EUR and DEM securities are retained, and DEM observations are converted using the fixed historical conversion `1 EUR = 1.95583 DEM`. No external FX series is used. Daily returns are retained for volatility calculations, while monthly month-end observations are used for the main run-up and crash analysis.

The daily return data were cleaned for obvious reversal and data errors. An observation is flagged when the current return or prior return exceeds 100 percent and the two-day compounded return is below 20 percent; any current return above 200 percent is also flagged. Flagged observations are set to missing, and the task here is to document that procedure rather than reproduce it in Python.


## 6. Monthly Returns and Industry Portfolio Construction

Monthly industry portfolios are value-weighted using lagged market capitalization based on `cshoc`. The benchmark market portfolio is constructed from the same German securities and the same monthly panel. This design keeps the portfolio returns comparable to the paper’s industry and market logic while remaining faithful to the executed German sample.


## 7. Identification of Large Price Increases

The main strict run-up identification requires a large industry run-up to satisfy three conditions: past 24-month raw return at least X; past 24-month net-of-market return at least X; and past 60-month raw return at least 50 percent. The signal is lagged one month to avoid look-ahead bias, only the first qualifying month is retained, and after an event is identified the following 24 months are suppressed to avoid repeated counting.

The relevant thresholds are 50%, 75%, 100%, 125%, and 150%. This strict rule is the basis for the main crash analysis, event-time analysis, and the Footnote-15-type regressions. A separate raw-only Table 1 definition is used for the paper-style table and is discussed in the later table section.


## 8. Subsequent Crash Outcomes

A crash is defined as a 40 percent drawdown from a previous peak during the following 24 months. The executed dataset records complete 12-month and 24-month outcome windows, forward raw returns, maximum drawdown, and both raw and net-market crash indicators. The evaluation therefore distinguishes between a large run-up and the behavior of the same industry after the signal is observed.


## 9. Pre-Run-Up Volatility

Pre-run-up annualized industry volatility is constructed from monthly industry returns and merged into the analysis sample. This allows the later regressions to test whether prior large returns remain predictive after conditioning on recent volatility, consistent with the paper’s empirical specification.


## 10. Analysis Sample and Merge

The executed volatility data and crash outcome files are merged into the main German industry analysis dataset. The merge is part of the executed Stata workflow; the notebook reads the final merged file for validation and reporting, without reconstructing the merge in Python.


In [ ]:
def read_result_excel(filename: str):
    path = PROJECT_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Required executed result file not found: {filename}')
    return pd.read_excel(path)

crash_results = read_result_excel('germany_crash_results.xlsx')
display(crash_results)

if not crash_results.empty:
    first_col = crash_results.columns[0]
    match = crash_results[crash_results[first_col].astype(str).str.contains('100', na=False)].head(1)
    if not match.empty:
        display(Markdown('The executed crash table is read directly from the project folder; the 100 percent threshold row is used below as the live specification reference.'))
        display(match)


## 11. Crash Probabilities Following Large Price Increases

The simplest empirical test asks whether large run-ups are followed by unusually high crash risk. The executed crash-outcome file reports the raw crash probability for each threshold and the associated forward returns and drawdowns, so the notebook reads and displays those results directly.


In [ ]:
crash_test = read_result_excel('germany_crash_test.xlsx')
display(crash_test)


## 12. Conditional versus Unconditional Crash Risk

The key question is whether crash risk after a large run-up exceeds the unconditional risk in the full German sample. This is the central comparison of the replication: the strict 100 percent episodes are evaluated against the broader eligible panel, and the notebook reads the executed test statistics rather than recomputing the formal test.


## 13. Statistical Tests of Crash Risk

The formal test records the unconditional crash probability, the probability after a 100 percent run-up, and the corresponding z-statistics and Fisher exact p-values. The notebook reads the executed test output and interprets the evidence in the language of the original paper.


In [ ]:
footnote15 = read_result_excel('germany_footnote15_regressions.xlsx')
display(footnote15)


## 14. Price Dynamics Around Large Run-Ups

The event study is based on the strict 100 percent run-up sample. Event month 0 is the first identified run-up month, and the executed window spans -24 to +48 months. The notebook reads the executed event-time file and displays the saved figure, without recalculating the event-study path.


In [ ]:
event_time = read_result_excel('germany_event_time_paths.xlsx')
display(event_time)
img_path = PROJECT_DIR / 'germany_event_time_paths.png'
if not img_path.exists():
    raise FileNotFoundError('Required executed result file not found: germany_event_time_paths.png')
display(Image(str(img_path)))


## 15. Large Price Increases and Subsequent Performance

The event-study and crash-outcome evidence together show whether large run-ups are followed by unusually weak subsequent market performance. This section interprets the combination of forward raw returns, maximum drawdowns, and crash outcomes from the executed outputs.


## 16. Threshold Results

The threshold results summarize the full range of large-run-up tests in the German sample. The project uses the executed table rather than a manually recreated result matrix, and risk-free dependent fields are intentionally reported as N/A because the project is restricted to a single WRDS source.


In [ ]:
table3_main = read_result_excel('germany_table3_main.xlsx')
display(table3_main)


## 17. Replication and Integrity Checks

The final set of replication checks provides a compact validation of the project integrity. It confirms that the expected threshold counts, the number of strict 100 percent run-ups, the event-time coverage, and the regression count are all consistent with the executed Stata workflow.


In [ ]:
replication_checks = read_result_excel('germany_replication_checks.xlsx')
display(replication_checks)


## 18. Validation of Executed Intermediate Files

This section briefly inspects the executed intermediate files to verify that the project is internally coherent. The notebook does not reconstruct the data-generation pipeline; it simply reads the executed datasets to confirm that the relevant monthly, volatility, and analysis files are present and usable.


In [ ]:
for name in [
    'germany_daily_clean.dta',
    'germany_industry_monthly.dta',
    'germany_runup_monthly.dta',
    'germany_runup_outcomes.dta',
    'germany_industry_volatility.dta',
    'germany_gsy_analysis.dta',
]:
    path = PROJECT_DIR / name
    print(f'{name}: {"FOUND" if path.exists() else "MISSING"}')


## 19. Portability and Reproduction Instructions

To reproduce the notebook on another PC, place the executed project files in the same folder as the notebook and run the file from that directory. The notebook discovers the project location by searching nearby folders for the expected execution outputs. Because the runtime files are read from disk at execution time, changes to the project data are reflected automatically when the notebook is rerun.


## 20. Limitations

This project is subject to several important limitations. First, the empirical work is restricted to a single WRDS source, and therefore no external risk-free series is used; risk-free dependent fields are reported as N/A. Second, DEM observations are converted using the fixed conversion `1 EUR = 1.95583 DEM`. Third, the strict German 100 percent run-up sample is small, which limits statistical power. Fourth, the notebook is a reporting layer: the Stata workflow is the raw-data calculation engine, and this notebook reads the executed outputs rather than re-estimating the underlying empirical pipeline. Finally, the paper-style Table 1 uses a raw-only 100 percent definition, whereas the strict main analysis uses the raw + net-market + five-year conditions.


## Table 1: paper-style raw-only definition

The paper-style Table 1 is based on a raw-only definition of a large run-up: a 24-month raw return above 100 percent and a 60-month raw return above 50 percent, with the same lagging, first-qualifying-month rule and 24-month suppression rule. This differs from the strict main-analysis definition, which also imposes a 24-month net-of-market requirement. The notebook uses the executed `germany_table1_paper_style.xlsx` file rather than the earlier, mistaken file that used the strict five-episode sample.


In [ ]:
table1 = read_result_excel('germany_table1_paper_style.xlsx')
display(table1)

display(Markdown('Table 1 is intentionally read from the executed paper-style file and is not recreated manually.'))
